In [1]:
#Setup parameters

params = {
    'data_folder': '../data', #Root folder for all data
    'annotation_folder': 'annotated', #Subfolder with annotation JSONs
    'tp_name': 'Mutable Annotations', #Name of layer in JSON that contains the true positive cells
    'mesh_regions': [771, 997], #ID for the largest region to subset (i.e. Pons) and brain boundary
    'mesh_root': '../code/ccf_files/CCF_meshes/json_verts_float', # location of the mesh files
    'ccf_res': 25, # Resolution of mesh in ccf space
    'template_res': [14.4, 14.4, 16], # resolution of LS template
    'ccf_template': '../data/lightsheet_template_ccf_registration/ccf_average_template_25.nii.gz',
    'smartspim_template': '../data/lightsheet_template_ccf_registration/smartspim_lca_template_25.nii.gz',
    'ccf_transforms': [
        '../data/lightsheet_template_ccf_registration/spim_template_to_ccf_syn_1Warp.nii.gz',
        '../data/lightsheet_template_ccf_registration/spim_template_to_ccf_syn_0GenericAffine.mat'
    ]
}


In [2]:
#Collect folder locations and metadata for datasets
import os
from glob import glob
from natsort import natsorted
from utils import json_to_xml, utils

folders = glob(os.path.join(params['data_folder'], 'SmartSPIM_*'))
data = {}
for folder in folders:
    lt_id = folder.split('_')[1]
    
    register_folder =  natsorted(glob(os.path.join(folder, 'image_atlas_alignment', 'Ex_*_Em_*')))[-1]
    template_transforms = [
        f"{register_folder}/ls_to_template_SyN_1Warp.nii.gz",
        f"{register_folder}/ls_to_template_SyN_0GenericAffine.mat"
    ]
    
    res_path = natsorted(glob(os.path.join(folder, 'image_tile_fusing', 'OMEZarr', 'Ex_*_Em_*.zarr/0/.zarray')))[-1]
    input_res = json_to_xml.read_json_as_dict(res_path)["shape"]

    input_dims = [
        input_res[-1],
        input_res[-2],
        input_res[-3],
    ]
    
    try:
        annotation_file = glob(os.path.join(params['data_folder'], params['annotation_folder'], f'{lt_id}*.json'))[0]
        annotation_data = json_to_xml.read_json_as_dict(annotation_file)
    except:
        print(f"Need to add annotation JSON for dataset {lt_id}.")
        continue
    
    cells = []
    for layers in annotation_data['layers']:
        if layers['name'] == params['tp_name']:
            for cell in layers['annotations']:
                cells.append(
                    [
                        cell['point'][0],
                        cell['point'][1],
                        cell['point'][2]
                    ]
                )
                
    acquisition = json_to_xml.read_json_as_dict(os.path.join(folder, 'acquisition.json'))
    
    data[lt_id] = {
        "template_transforms": template_transforms,
        "annotation_file": annotation_file,
        "orientation": acquisition['axes'],
        "input_dims": input_dims,
        "cell_locations": cells,
        "ng_file": annotation_data,
    }

In [3]:
#Functions to warp CCF Mesh
import json
import numpy as np
import pickle

def orient_mesh(
    vert_list,
    reg_dims: list,
    ds: int,
    orient: str,
    orient_matrix: np.ndarray,
    institute: str,
):
    """
    Imports cell locations from a XML file of cell likelihoods.

    Parameters
    -------------
    cell_likelihoods_path: str or PathLike
        Path to the cell likelihoods XML file.
    reg_dims: list
        Resolution (pixels) of the image used for segmentation, ordered relative to zarr.
    ds: int
        Factor by which the image for registration was downsampled from input_dims.
    orient: str
        The orientation the brain was imaged.
    orient_matrix: np.ndarray
        The direction of the axis of input cells relative to registration
    institute: str
        The institution that imaged the dataset.

    Returns
    -------------
    np.ndarray
        Array with cell locations scaled and oriented
    """
    cells = []

    for row in vert_list:
        
        x, y, z = int(row[2]), int(row[1]), int(row[0])
    
        # Corrects for a bug in acquisition as SPL is not an actual imaging orientation
        if orient == "sal":
            y = reg_dims[1] - (y / ds)
        else:
            y = y / ds
    
        cells.append(
                (z / ds, y, x / ds)
            )
    
    cells = np.array(cells)
    
    for idx, dim_orient in enumerate(orient_matrix.sum(axis = 1)):
        if dim_orient < 0:
            cells[:, idx] = reg_dims[idx] - cells[:, idx]

    return cells


def get_orientation(params: dict) -> str:
    """
    Fetch aquisition orientation to identify origin for cell locations
    from cellfinder. Important for read_xml function in quantification
    script

    Parameters
    ----------
    params : dict
        The orientation information from processing_manifest.json

    Returns
    -------
    orient : str
        string that indicates axes order and direction current available
        options are:
            'spr'
            'sal'
        But more may be used later
    """

    orient = ["", "", ""]
    for vals in params:
        direction = vals["direction"].lower()
        dim = vals["dimension"]
        orient[dim] = direction[0]

    return "".join(orient)

def get_orientation_transform(orientation_in: str, orientation_out: str) -> tuple:
    """
    Takes orientation acronyms (i.e. spr) and creates a convertion matrix for
    converting from one to another

    Parameters
    ----------
    orientation_in : str
        the current orientation of image or cells (i.e. spr)
    orientation_out : str
        the orientation that you want to convert the image or
        cells to (i.e. ras)

    Returns
    -------
    tuple
        the location of the values in the identity matrix with values
        (original, swapped)
    """

    reverse_dict = {"r": "l", "l": "r", "a": "p", "p": "a", "s": "i", "i": "s"}

    input_dict = {dim.lower(): c for c, dim in enumerate(orientation_in)}
    output_dict = {dim.lower(): c for c, dim in enumerate(orientation_out)}

    transform_matrix = np.zeros((3, 3))
    for k, v in input_dict.items():
        if k in output_dict.keys():
            transform_matrix[v, output_dict[k]] = 1
        else:
            k_reverse = reverse_dict[k]
            transform_matrix[v, output_dict[k_reverse]] = -1

    if orientation_in.lower() == "spl" or orientation_out.lower() == "spl":
        transform_matrix = abs(transform_matrix)

    original, swapped = np.where(transform_matrix.T)

    return original, swapped, transform_matrix

def get_region_lists():
    """
    Import list of acronyms of brain regions
    """
    
    CCF_dir = '../code/ccf_files/CCF_meshes'
    
    
    # Reading non-crossing structures to get acronyms
    with open(os.path.join(CCF_dir, "non_crossing_structures"), "rb") as f:
        hemi_struct = pickle.load(f)
        hemi_struct.remove(1051)  # don't know why this is being done
        hemi_labeled = [(s, "hemi") for s in hemi_struct]

    # Reading mid-crossing structures to get acronyms
    with open(os.path.join(CCF_dir, "mid_crossing_structures"), "rb") as f:
        u = pickle._Unpickler(f)
        u.encoding = "latin1"
        mid_struct = u.load()
        mid_labeled = [(s, "mid") for s in mid_struct]

    return hemi_labeled + mid_labeled

def load_json_mesh(root, struct):
    """
    Loads meshes that are stored in json files like the CCF meshes

    Parameters
    ----------
    root : str
        path to mesh

    Returns
    -------
    tuple
        The vertices and faces of a given mesh

    """
        
    region_metadata = get_region_lists()
    region_dict = {i[0]: i[1] for i in region_metadata}
    fname = os.path.join(root, f"{struct}.json")
        
    with open(fname) as f:
        structure_data = json.loads(f.read())
        verts, faces = (
            np.array(structure_data[struct]["vertices"]),
            np.array(structure_data[struct]["faces"]),
        )
            
        if region_dict[int(struct)] == 'hemi':
            offset = faces.max() + 1
            faces = np.vstack((faces, faces + offset))
                
            verts_2 = verts.copy()
            verts_2[:, 0] = verts_2[:, 0] + (5700 - verts_2[:, 0]) * 2
            verts = np.vstack((verts, verts_2))
            
    return verts, faces

def warp_mesh(verts, faces, ccf_template, smartspim_template, ccf_transforms, template_transforms):
        
    # Transform to template
    ccf_params = utils.get_template_info(ccf_template)
    ants_verts = utils.convert_to_ants_space(ccf_params, verts)
    template_verts = utils.apply_transforms_to_points(
        ants_verts, ccf_transforms, invert=(False, False)
    )

    # Transform to lightsheet
    template_params = utils.get_template_info(smartspim_template)
    ls_verts = utils.apply_transforms_to_points(
        template_verts,
        template_transforms,
        invert=(False, False),
    )
    converted_verts = utils.convert_from_ants_space(template_params, ls_verts)
        
    return converted_verts, faces

def add_annotation_layer(ng_dict, cells):
    
    ng_cells = []
    for c, cell in enumerate(cells):
        ng_cells.append(
            {
                "point": [float(cell[0]), float(cell[1]), float(cell[2]), 0.5],
                "type": "point",
                "id": f"cell_{int(c)}"
            }
        )
    
    cropped_layer = {
        "type": "annotation",
        "source": {
            "url": "local://annotations",
            "transform": {
                "outputDimensions": {
                    "z": [0.000002, "m"],
                    "y": [0.0000018, "m"],
                    "x": [0.0000018, "m"],
                    "t": [0.001, "s"],
                },
            },
            
        },
        "tool": "annotatePoint",
        "tab": "annotations",
        "annotationColor": "#ff0000",
        "annotations": ng_cells,
        "name": "Pons Cells"
    }
    
    ng_dict['layers'].append(cropped_layer)
    return ng_dict


def dilate_mesh(mesh, dilation):
    
    if not isinstance(dilation, list):
        dilation = [dilation] * 3
        
    com = mesh.center_of_mass()
    
    verts, faces = mesh.points(), mesh.faces()
    
    for c, v in enumerate(verts):
        for i in range(3):
            v[i] = (v[i] - com[i]) * dilation[i] + com[i]
        verts[c, :] = v
    
    return vedo.Mesh([verts, faces])

class NumpyTypeEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.generic):
            return obj.item()
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        return json.JSONEncoder.default(self, obj)

def save_json(data, file_name):
    
    with open(file_name, 'w') as fp:
        json.dump(data, fp, indent = 2, cls=NumpyTypeEncoder)
    

In [4]:
# transform parent mesh for each dataset and gab only cells within mesh
import os
import numpy as np
import pandas as pd
import vedo
import json

#dimensions are DV, AP, ML
dilation = [1.0, 1.3, 1.0]

JSON_PATH = '../results/jsons'
CSV_PATH = '../results/csvs'

if not os.path.exists(JSON_PATH):
    os.mkdir(JSON_PATH)
    
if not os.path.exists(CSV_PATH):
    os.mkdir(CSV_PATH)

for lt_id, dataset in data.items():
    
    meshes = {}
    
    for struct in params['mesh_regions']:
    
        verts, faces = load_json_mesh(params['mesh_root'], str(struct))
        verts = verts[:, [0, 2, 1]] / params['ccf_res']
    
        verts_ls, faces_ls = warp_mesh(
            verts,
            faces,
            params['ccf_template'],
            params['smartspim_template'],
            params['ccf_transforms'],
            dataset['template_transforms']
        )
    
        scale = [params['ccf_res']/dim for dim in [14.4, 14.4, 16]]
        for i in range(3):
            verts_ls[:, i] = verts_ls[:, i] * scale[i]
            
        verts_ls *= 2**3
        
        orientation = get_orientation(dataset['orientation'])
        _, swapped, mat = get_orientation_transform('ras', orientation)

        verts_orient = orient_mesh(
            vert_list=verts_ls, 
            reg_dims=data[lt_id]['input_dims'],
            ds=1,
            orient='ras',
            orient_matrix=mat,
            institute='AIND',
        )

        verts_orient = verts_orient[:, swapped]
        
        
        if struct != 997:
            
            print(f'Getting cells for {struct} in dataset {lt_id}')

            hull = vedo.ConvexHull(verts_orient)
            dilated_hull = dilate_mesh(hull, dilation)
            
            meshes['hull'] = {
                'faces': dilated_hull.faces(),
                'verts': dilated_hull.points()
                
            }
            
            meshes[struct] = {
                'faces': faces_ls,
                'verts': verts_orient
            }

            locations = dilated_hull.inside_points(dataset['cell_locations']).points()
            print(f"started with {len(dataset['cell_locations'])} cells and ended with {len(locations)} cells.")
            internal_cells = [cell for cell in locations]
            meshes[struct]['internal_cells'] = internal_cells
            
            df = pd.DataFrame(internal_cells, columns = ['Z', 'Y', 'X'])
            df.to_csv(f"{CSV_PATH}/{str(lt_id)}_pons_cells.csv")
            
            df = pd.DataFrame(dataset['cell_locations'], columns = ['Z', 'Y', 'X'])
            df.to_csv(f"{CSV_PATH}/{str(lt_id)}_all_cells.csv")
            
            new_ng_link = add_annotation_layer(dataset['ng_file'], locations)
            save_json(new_ng_link, f"{JSON_PATH}/{str(lt_id)}_new.json")
        else:
            meshes[struct] = {
                'faces': faces_ls,
                'verts': verts_orient
            }
        
    data[lt_id]['meshes'] = meshes

Getting cells for 771 in dataset 751017


2025-04-23 23:40:06.785 ( 106.746s) [    7F367F268340]      vtkDelaunay3D.cxx:513   WARN| vtkDelaunay3D (0x55d8258b4ba0): 44 degenerate triangles encountered, mesh quality suspect


started with 2687 cells and ended with 693 cells.
Getting cells for 771 in dataset 755069


2025-04-23 23:42:30.802 ( 250.762s) [    7F367F268340]      vtkDelaunay3D.cxx:513   WARN| vtkDelaunay3D (0x55d8258c1fa0): 52 degenerate triangles encountered, mesh quality suspect


started with 678 cells and ended with 653 cells.
Getting cells for 771 in dataset 762196


...

started with 3775 cells and ended with 476 cells.
Getting cells for 771 in dataset 755072


2025-04-23 23:51:56.357 ( 816.318s) [    7F367F268340]      vtkDelaunay3D.cxx:513   WARN| vtkDelaunay3D (0x55d827039e50): 42 degenerate triangles encountered, mesh quality suspect


started with 797 cells and ended with 774 cells.
Getting cells for 771 in dataset 751019


2025-04-23 23:54:16.904 ( 956.865s) [    7F367F268340]      vtkDelaunay3D.cxx:513   WARN| vtkDelaunay3D (0x55d826d40520): 52 degenerate triangles encountered, mesh quality suspect


started with 2960 cells and ended with 665 cells.


In [7]:
import k3d

# plot example of cropped cells
lt_id = '755073'
mesh_id = 771


def rgb_to_hex(r,g,b):
    # Convert to a hexadecimal string
    hex_color = f'{r:02x}{g:02x}{b:02x}'
    # Convert the hexadecimal string to an integer in base-16
    color_int = int(hex_color, 16)
    return color_int

# Initialize plot
plot = k3d.plot()

## Plot meshes
brain_mesh = data[lt_id]['meshes'][997]
mesh = k3d.mesh(brain_mesh['verts'], brain_mesh['faces'], color=rgb_to_hex(128,128,128), opacity=0.05)
plot += mesh

region_mesh = data[lt_id]['meshes'][mesh_id]
mesh = k3d.mesh(region_mesh['verts'], region_mesh['faces'], color =rgb_to_hex(0,255, 255), opacity = 0.3)
plot += mesh

hull_mesh = data[lt_id]['meshes']['hull']
mesh = k3d.mesh(hull_mesh['verts'], hull_mesh['faces'], color =rgb_to_hex(255, 0, 0), opacity = 0.1)
plot += mesh

# plot annotations
region_pts = np.array(data[lt_id]["cell_locations"])
pts = k3d.points(positions = region_pts.astype('float32'), point_size = 25, color = rgb_to_hex(0, 0, 0))
plot += pts

# plot annotations
region_pts = np.array(data[lt_id]['meshes'][mesh_id]['internal_cells'])
pts = k3d.points(positions = region_pts.astype('float32'), point_size = 25, color = rgb_to_hex(255, 0, 0))
plot += pts



plot.display()

/opt/conda/lib/python3.9/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "float64" does not match required type "float32". A coerced copy has been created.
  warnings.warn(
/opt/conda/lib/python3.9/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "int64" does not match required type "uint32". A coerced copy has been created.
  warnings.warn(


Output()

In [ ]:
# region_mesh = data[lt_id]['meshes'][mesh_id]
new_mesh = vedo.Mesh([region_mesh['verts'], region_mesh['faces']])
new_mesh = new_mesh.decimate(0.2)

plot = k3d.plot()

mesh = k3d.mesh(region_mesh['verts'], region_mesh['faces'], color =rgb_to_hex(255,0, 0), opacity = 0.3)
plot += mesh

mesh = k3d.mesh(new_mesh.points(), new_mesh.faces(), color =rgb_to_hex(0,0, 255), opacity = 0.3)
plot += mesh

plot.display()

In [7]:
new_mesh = vedo.Mesh([region_mesh['verts'], region_mesh['faces']])
volume = new_mesh.binarize(
    spacing=[1.8, 2.0, 1.8],
)

volume.dilate(neighbours=(5, 5, 5))
iso = volume.isosurface().decimate()

v = iso.points()
f = iso.faces()

plot = k3d.plot()

mesh = k3d.mesh(v, f, color =rgb_to_hex(0,0, 255), opacity = 0.3)
plot += mesh

plot.display()

Output()